# Principle 1: Preferential Attachment

## Experiment Configuration

Set models, simulation sizes, temperatures, environments, and CoT options here before running the experiment cells.

### Configuration Cell

このセルで実験条件を一括管理します。`EXPERIMENTS` の各要素が1つの実験条件で、`model`、`environment`、`degrees_experiment`、`COT`、`parameters` を同じ辞書にまとめています。`parameters` は `run_network_formation_experiment(...)` に渡す `n_min`、`n_max`、`n_step`、`num_simulations` です。


In [ ]:
# Shared settings.
from utils import get_shared_experiment_settings

_SHARED_SETTINGS = get_shared_experiment_settings()
BASELINE_MODEL = _SHARED_SETTINGS['BASELINE_MODEL']
MODEL_NAMES = _SHARED_SETTINGS['MODEL_NAMES']
DEFAULT_TEMPERATURES = _SHARED_SETTINGS['DEFAULT_TEMPERATURES']
DEFAULT_COT_CONFIG = _SHARED_SETTINGS['DEFAULT_COT_CONFIG']
COT_RETRY_MAX_NEW_TOKENS = _SHARED_SETTINGS['COT_RETRY_MAX_NEW_TOKENS']
ENVIRONMENTS = [
    ('school', 'classmates'),
    ('work', 'colleagues'),
    ('community', 'neighbors'),
]

# Each dictionary is one runnable experiment condition.
# environment=None means the baseline prompt without a school/work/community context.
EXPERIMENTS = [
    # Model comparison without CoT under the baseline prompt.
    *[
        {
            'name': f'model_{model.replace("/", "-")}_baseline',
            'model': model,
            'environment': None,
            'degrees_experiment': False,
            'COT': False,
            'temperatures': DEFAULT_TEMPERATURES,
            'parameters': dict(n_min=50, n_max=50, n_step=1, num_simulations=1),
            'analyze_detail': model == BASELINE_MODEL,
        }
        for model in MODEL_NAMES
    ],

    # Environment comparison without CoT.
    *[
        {
            'name': f'model_{BASELINE_MODEL.replace("/", "-")}_{environment[0]}_{environment[1]}',
            'model': BASELINE_MODEL,
            'environment': environment,
            'degrees_experiment': False,
            'COT': False,
            'temperatures': DEFAULT_TEMPERATURES,
            'parameters': dict(n_min=50, n_max=50, n_step=1, num_simulations=1),
        }
        for environment in ENVIRONMENTS
    ],

    # Direct degree-count condition without CoT.
    {
        'name': f'model_{BASELINE_MODEL.replace("/", "-")}_baseline_degree',
        'model': BASELINE_MODEL,
        'environment': None,
        'degrees_experiment': True,
        'COT': False,
        'temperatures': DEFAULT_TEMPERATURES,
        'parameters': dict(n_min=50, n_max=50, n_step=1, num_simulations=1),
        'include_in_exponent_summary': False,
    },

    # Model comparison with CoT / thinking under the baseline prompt.
    {
        'name': f'model_{BASELINE_MODEL.replace("/", "-")}_baseline_cot',
        'model': BASELINE_MODEL,
        'environment': None,
        'degrees_experiment': False,
        'COT': True,
        'cot_config': DEFAULT_COT_CONFIG,
        'temperatures': DEFAULT_TEMPERATURES,
        'parameters': dict(n_min=50, n_max=50, n_step=1, num_simulations=1),
    }
]

RUN_EXPERIMENTS = True
RUN_ANALYSIS = True


### Python Version Check

This cell prints the active Python version. It is a quick environment sanity check before installing dependencies and running the notebook.

In [ ]:
!python -V

### Colab Repository Setup

This cell clones or updates the repository in local Colab storage, mounts Google Drive, creates a persistent output directory in Drive, and installs `requirements.txt`. Keeping the working directory under `/content` avoids Google Drive mount disconnect errors during imports while results are saved under `/content/drive/MyDrive/llm-network-formation-outputs`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

REPO_DIR = Path('/content/llm-network-formation')
OUTPUT_DIR = Path('/content/drive/MyDrive/llm-network-formation-outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if 'DEFAULT_PROFILES_FILENAME' in globals():
    DEFAULT_PROFILES_FILENAME = str(OUTPUT_DIR / 'profiles.jsonl')
if 'LUCKY_NUMBER_PROFILES_FILENAME' in globals():
    LUCKY_NUMBER_PROFILES_FILENAME = str(OUTPUT_DIR / 'profiles_with_lucky_number.jsonl')
if 'EXPERIMENTS' in globals():
    for experiment in EXPERIMENTS:
        profile_path = experiment.get('profiles_filename')
        if profile_path is not None and Path(profile_path).name == 'profiles.jsonl':
            experiment['profiles_filename'] = str(OUTPUT_DIR / 'profiles.jsonl')
        if profile_path is not None and Path(profile_path).name == 'profiles_with_lucky_number.jsonl':
            experiment['profiles_filename'] = str(OUTPUT_DIR / 'profiles_with_lucky_number.jsonl')



if REPO_DIR.exists():
    %cd $REPO_DIR
    !git pull --ff-only
else:
    %cd /content
    !git clone https://github.com/yohei-kobashi/llm-network-formation.git
    %cd $REPO_DIR

!pip uninstall -y vllm transformers torch torchvision torchaudio xformers
!pip install -U pip setuptools wheel
!pip install --no-cache-dir 'numpy==2.0.2' 'scipy==1.14.1'
!pip install matplotlib networkx pandas seaborn netgraph powerlaw openai anthropic replicate
!pip install -U vllm --extra-index-url https://download.pytorch.org/whl/cu128
!pip install -U git+https://github.com/huggingface/transformers.git accelerate sentencepiece
!python -c "import numpy, scipy, netgraph, vllm, transformers; print('dependency check ok', numpy.__version__, scipy.__version__, vllm.__version__, transformers.__version__)"

Cloning into 'llm-network-formation'...
remote: Enumerating objects: 372, done.
remote: Counting objects: 100% (265/265), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 372 (delta 156), reused 221 (delta 113), pack-reused 107 (from 1)
Receiving objects: 100% (372/372), 143.57 MiB | 13.88 MiB/s, done.
Resolving deltas: 100% (196/196), done.
/content/llm-network-formation
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.7/96.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.2/458.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 372.3/372.3 kB 40.8 MB/s eta 0:00:00


### API Credentials

This cell reads secrets from Colab userdata and sets environment variables for OpenAI and Hugging Face access. `OPENAI_API_KEY` is required for GPT models; `HF_TOKEN` is optional but useful for Hugging Face model downloads.

In [ ]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
if userdata.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

### Helper Functions and Analysis Code

This cell imports the scientific Python stack and defines the network formation, LLM neighbor selection, graph reconstruction, and analysis helpers. The main experiment cells below call `run_network_formation_experiment(...)`, `analyze_experiments(...)`, and `analyze_experiments_multiple_llms(...)` from this cell.

In [ ]:
import json
import os
import collections

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from utils import (
    filter_supported_models,
    get_response,
    get_responses,
    summarize_reasons,
    set_principle1_runtime_options,
    build_response_schema,
    build_response_list_schema,
    build_acceptance_response_schema,
    first_json_object,
    first_json_array,
    normalize_name,
    principle1_draw_graph as draw_graph,
    principle1_initialize_candidate_state as initialize_candidate_state,
    principle1_update_candidate_state as update_candidate_state,
    principle1_build_prompt_candidates as build_prompt_candidates,
    principle1_network_growth as network_growth,
    principle1_select_neighbor as select_neighbor,
    principle1_build_neighbor_request as build_neighbor_request,
    principle1_parse_neighbor_response as parse_neighbor_response,
    recover_name_from_malformed_response,
    print_llm_parse_error,
    principle1_initialize_growth_state as initialize_growth_state,
    principle1_advance_growth_state as advance_growth_state,
    principle1_write_growth_state as write_growth_state,
    principle1_pending_growth_states_for_experiment as pending_growth_states_for_experiment,
    principle1_run_network_formation_experiments_batch as run_network_formation_experiments_batch,
    principle1_run_network_formation_experiment as run_network_formation_experiment,
    principle1_reconstruct_graphs as reconstruct_graphs,
    principle1_analyze_experiments as analyze_experiments,
    principle1_analyze_experiments_multiple_llms as analyze_experiments_multiple_llms,
    principle1_run_configured_experiments,
)

MEDIUM_SIZE = 24
SMALL_SIZE = 0.85 * MEDIUM_SIZE
BIGGER_SIZE = 1.5 * MEDIUM_SIZE
set_principle1_runtime_options(
    baseline_model=BASELINE_MODEL,
    medium_size=MEDIUM_SIZE,
)

print("Principle 1 helper functions loaded from utils.py.")


## Network Formation Experiments


### Configured Network Formation Experiments

このセルは `EXPERIMENTS` を順に読み、各辞書の設定に従って実験を実行します。出力ファイル名は `name` から `OUTPUT_DIR / principle_1_{name}.jsonl` として生成し、非CoTとCoTの結果を分けて分析します。


In [ ]:
PRINCIPLE1_RUN_RESULTS = principle1_run_configured_experiments(
    EXPERIMENTS,
    OUTPUT_DIR,
    DEFAULT_TEMPERATURES,
    run_experiments=RUN_EXPERIMENTS,
    run_analysis=RUN_ANALYSIS,
)

SUPPORTED_MODELS = PRINCIPLE1_RUN_RESULTS['supported_models']
non_cot_outfiles = PRINCIPLE1_RUN_RESULTS['non_cot_outfiles']
cot_outfiles = PRINCIPLE1_RUN_RESULTS['cot_outfiles']
experiments_to_analyze = PRINCIPLE1_RUN_RESULTS['experiments_to_analyze']
experiment_records = PRINCIPLE1_RUN_RESULTS['experiment_records']
